In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file_path = str(PROJECT_ROOT / "Dataset Specs" / "hospital_occupancy_daily.csv")

df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

daily_occupancy = df["occupied_beds"].asfreq("D")

print(daily_occupancy.head())
print(daily_occupancy.index.freq)

In [ ]:
train_size = int(len(daily_occupancy) * 0.7)

train = daily_occupancy.iloc[:train_size]
test = daily_occupancy.iloc[train_size:]

print("Train:", train.index.min(), "to", train.index.max(),
      "len =", len(train))
print("Test :", test.index.min(), "to", test.index.max(),
      "len =", len(test))

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    train,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

sarima_forecast = sarima_fit.forecast(steps=len(test))
sarima_pred = pd.Series(sarima_forecast.to_numpy(), index=test.index)

rmse = np.sqrt(mean_squared_error(test, sarima_pred))
mae = mean_absolute_error(test, sarima_pred)
mape = np.mean(np.abs((test - sarima_pred) / test)) * 100

print("SARIMA performance on test set:")
print(f"RMSE: {rmse:.3f} beds")
print(f"MAE:  {mae:.3f} beds")
print(f"MAPE: {mape:.3f}%")

In [ ]:
output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/70_30"
)
output_dir.mkdir(parents=True, exist_ok=True)

sarima_df = pd.DataFrame({
    "date": test.index,
    "actual_occupied_beds": test.to_numpy(),
    "sarima_predicted_occupied_beds": sarima_pred.to_numpy()
})

sarima_df.to_csv(
    output_dir / "sarima_occupancy_predictions.csv",
    index=False
)

print(f"SARIMA predictions saved to: {output_dir}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test, label="Actual Occupancy",
         color="black", linewidth=1)
plt.plot(sarima_pred.index, sarima_pred,
         label="SARIMA Forecast", color="blue", linewidth=2)

plt.title("SARIMA: Actual vs Forecast Occupancy (70/30 Test Split)")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
sarima_rolling_pred = pd.Series(
    rolling_preds,
    index=test_series.index,
    name="sarima_rolling"
)

y_test = test_series

In [ ]:
rmse = np.sqrt(mean_squared_error(test_series, rolling_pred_series))
mae  = mean_absolute_error(test_series, rolling_pred_series)
mape = np.mean(np.abs((test_series - rolling_pred_series) / test_series)) * 100
accuracy = 100 - mape

print("Rolling SARIMA performance:")
print("RMSE:", rmse)
print("MAE :", mae)
print("MAPE:", mape)
print("Accuracy (100 - MAPE):", accuracy)

In [ ]:
# Plot Actual vs Rolling SARIMA
plt.figure(figsize=(12,5))
plt.plot(test_series.index, test_series, label="Actual", color="black")
plt.plot(sarima_rolling_pred.index, sarima_rolling_pred, 
         label="Rolling SARIMA", color="blue", alpha=0.8)

plt.title("Rolling SARIMA vs Actual (Test Period)")
plt.xlabel("Date")
plt.ylabel("Admissions")
plt.legend()
plt.show()

In [ ]:
sarima_rolling_df = pd.DataFrame({
    "date": y_test.index,
    "actual": y_test.values,
    "sarima_rolling_pred": sarima_rolling_pred.values
})

sarima_rolling_df.to_csv("../70_30 split/predictions/sarima_rolling_predictions.csv", index=False)

print("SARIMA rolling predictions saved successfully.")


In [ ]:
actual_df = pd.DataFrame({
    "date": y_test.index,
    "actual": y_test.values
})

actual_df.to_csv("../70_30 split/predictions/actual.csv", index=False)